In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

data_dir = Path('data')
enron_file = data_dir / 'enron_spam_data.zip'

raw_data = pd.read_csv(enron_file, compression='zip')
df = pd.DataFrame({
    'text': raw_data['Subject'].fillna('') + ' ' + raw_data['Message'].fillna(''),
    'label': (raw_data['Spam/Ham'] == 'spam').astype(int)
})

df['text'] = df['text'].str.strip()
df = df[df['text'].str.len() > 0].drop_duplicates(subset='text')

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Data ready. Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")

Data ready. Train: 24369, Test: 6093


In [2]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english', min_df=3, max_df=0.9)

print("Vectorizing training and testing texts...")
X_train_mat = vectorizer.fit_transform(X_train_raw)
X_test_mat = vectorizer.transform(X_test_raw)

print(f"Vocabulary size: {X_train_mat.shape[1]}")
print("Vectorization completed.")

Vectorizing training and testing texts...
Vocabulary size: 50319
Vectorization completed.


In [3]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB

online_model = MultinomialNB()
batch_size = 1000
classes = np.array([0, 1])

num_samples = X_train_mat.shape[0]
print(f"Starting online learning on {num_samples} samples...")

for i in range(0, num_samples, batch_size):
    X_batch = X_train_mat[i : i + batch_size]
    y_batch = y_train[i : i + batch_size]
    online_model.partial_fit(X_batch, y_batch, classes=classes)

print("Batch training completed.")

Starting online learning on 24369 samples...
Batch training completed.


In [4]:
from sklearn.metrics import classification_report

print("Evaluating online model on test data...")
y_pred = online_model.predict(X_test_mat)

print("\n" + "=" * 55)
print("Classification Report (Exercise 4: Online Learning)")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], digits=4))

Evaluating online model on test data...

Classification Report (Exercise 4: Online Learning)
              precision    recall  f1-score   support

         Ham     0.9893    0.9840    0.9866      3182
        Spam     0.9826    0.9883    0.9854      2911

    accuracy                         0.9860      6093
   macro avg     0.9859    0.9861    0.9860      6093
weighted avg     0.9861    0.9860    0.9861      6093

